# molprop-ensemble — Notebook Utama

Pipeline ensemble **ECFP+RF · ChemBERTa · D-MPNN** + SMILES-Enumeration TTA untuk prediksi properti molekul (BBBP, BACE, ClinTox).

### Cara pakai
1. Pastikan panel kanan: **Accelerator = GPU T4 x2**, **Internet = On**.
2. Repo privat → siapkan **Secret `GH_TOKEN`** (Add-ons → Secrets). Lihat `KAGGLE.md`.
3. Jalankan sel **Setup**, lalu atur **Konfigurasi** (panel widget — tanpa edit kode), lalu jalankan sisanya berurutan (atau *Run All* setelah konfigurasi disetel).

Urutan: **1) Setup → 2) Konfigurasi → 3) Data → 4) Jalankan → 5) Hasil → 6) Simpan.**

## 1. Setup — clone, install, cek environment

In [ ]:
# 1a. Clone repo dari GitHub (pakai GH_TOKEN bila repo privat)
REPO_OWNER, REPO_NAME, REPO_DIR = "belvahector-ship-it", "pharm_", "pharm_"
import os, subprocess
if not os.path.exists(REPO_DIR):
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    url = (f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if token
           else f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git")
    print("clone via GH_TOKEN..." if token else "GH_TOKEN tak ada -> coba clone publik.")
    r = subprocess.run(["git", "clone", url, REPO_DIR], capture_output=True, text=True,
                       timeout=120, stdin=subprocess.DEVNULL)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("Clone gagal — repo privat tanpa GH_TOKEN valid? Lihat KAGGLE.md.")
else:
    print(f"{REPO_DIR} sudah ada, skip clone.")
%cd {REPO_DIR}
!git pull --quiet && git log --oneline -1

In [ ]:
# 1b. Install dependency (chemprop 2.x; deepchem tidak diperlukan)
!pip install -q rdkit "chemprop>=2.1,<3.0" transformers
print("install selesai")

In [ ]:
# 1c. Cek environment (WAJIB 2 GPU) + versi paket
import torch
print("torch:", torch.__version__, "| GPU:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  GPU", i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, "Set Accelerator = GPU T4 x2 di panel kanan, lalu restart."
import rdkit, sklearn, transformers, chemprop
print("rdkit", rdkit.__version__, "| sklearn", sklearn.__version__,
      "| transformers", transformers.__version__, "| chemprop", chemprop.__version__)

## 2. Konfigurasi
Atur lewat panel di bawah — **tidak perlu mengedit kode**. Setelah menyetel, jalankan sel-sel di bawahnya (atau *Run All* lagi).

In [ ]:
# Panel konfigurasi interaktif (tanpa edit variabel manual)
try:
    import ipywidgets as W
    from IPython.display import display, HTML
    _HAS_W = True
except Exception:
    _HAS_W = False

if _HAS_W:
    display(HTML("<div style=\"border-left:5px solid #ff9800;background:#fff8e1;"
        "padding:12px 14px;border-radius:4px;font-size:14px\"><b>&#9881;&#65039; Konfigurasi run</b><br>"
        "Pilih di bawah lalu jalankan sel berikutnya. <b>Reset</b> wajib dicentang SEKALI "
        "setelah update kode/algoritma split berubah.</div>"))
    cfg_reset = W.Checkbox(value=False, description="Reset artefak lama (centang sekali setelah update)",
                           indent=False, layout=W.Layout(width="auto"))
    cfg_mode = W.RadioButtons(
        options=[("Trial cepat — 1 dataset x 1 seed (verifikasi pipeline)", "trial"),
                 ("Full run — 3 dataset x 5 seed (hasil paper, lama)", "full")],
        value="trial", description="Mode:", style={"description_width": "initial"},
        layout=W.Layout(width="auto"))
    cfg_ds = W.Dropdown(options=["bbbp", "bace", "clintox"], value="bbbp",
                        description="Trial dataset:", style={"description_width": "initial"})
    cfg_seed = W.Dropdown(options=[0, 1, 2, 3, 4], value=0,
                          description="Trial seed:", style={"description_width": "initial"})
    display(cfg_reset, cfg_mode, cfg_ds, cfg_seed)
else:
    class _V:
        def __init__(self, v): self.value = v
    cfg_reset, cfg_mode, cfg_ds, cfg_seed = _V(False), _V("trial"), _V("bbbp"), _V(0)
    print("ipywidgets tak tersedia -> default: mode=trial, dataset=bbbp, seed=0, reset=False.")

## 3. Data & scaffold split

In [ ]:
# 3a. Reset artefak (hanya bila dicentang di Konfigurasi)
import os, shutil
if cfg_reset.value:
    for d in ["data/splits", "outputs/predictions", "outputs/checkpoints", "outputs/results"]:
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)
    print("\u2713 Artefak lama dibersihkan (split/prediksi/checkpoint).")
else:
    print("Reset dilewati (kotak Reset tidak dicentang).")

In [ ]:
# 3b. Fase 1 — scaffold split (fixed, dipakai ulang semua model)
!python scripts/01_prepare_data.py

In [ ]:
# 3c. Diagnostik mutu split — WAJIB "overlap SCAFFOLD = 0" & "bocor->train = 0.0%"
!python scripts/00_diagnose_split.py

## 4. Jalankan pipeline
Satu alur: **training (Fase 4) → TTA (5) → fusion (6) → evaluasi (7)**. Mengikuti pilihan Mode di Konfigurasi. ChemBERTa→GPU0, D-MPNN→GPU1, RF→CPU (paralel). Checkpoint + resume aktif.

In [ ]:
import subprocess
full = (cfg_mode.value == "full")
extra = [] if full else ["--datasets", cfg_ds.value, "--seeds", str(cfg_seed.value)]
print("MODE:", "FULL 3x5" if full else f"TRIAL {cfg_ds.value} seed {cfg_seed.value}")

# Fase 4 — training paralel di 2 GPU + CPU
procs = {m: subprocess.Popen(["python", "scripts/02_train_baselines.py", "--model", m] + extra)
         for m in ["chemberta", "dmpnn", "rf"]}
for name, p in procs.items():
    rc = p.wait()
    print(f"[Fase4 {name}] rc={rc}")
    assert rc == 0, f"training {name} gagal (rc={rc}) — cek log di atas"

# Fase 5-7
for script in ["04_run_tta.py", "03_run_fusion.py", "05_evaluate.py"]:
    rc = subprocess.run(["python", f"scripts/{script}"] + extra).returncode
    print(f"[{script}] rc={rc}")
    assert rc == 0, f"{script} gagal (rc={rc})"
print("\nPIPELINE SELESAI.")

## 5. Hasil
Tabel final (`roc_auc_mean±std`, bootstrap 95% CI, p-value vs baseline post-hoc, Cohen's d).
Catatan: kolom `p_value`/`std` baru bermakna di **Full run** (n=5); pada Trial (n=1) wajar NaN/0.

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", None, "display.width", 200)
pd.read_csv("outputs/results/final_table.csv")

## 6. Simpan hasil
`outputs/` di-gitignore (tak ikut GitHub). Zip di bawah lalu unduh dari panel Output, atau klik **Save Version**.

In [ ]:
!pip freeze > outputs/results/pip_freeze.txt
!cd outputs && zip -rq /kaggle/working/hasil_outputs.zip .
print("tersimpan: /kaggle/working/hasil_outputs.zip  &  outputs/results/{environment,pip_freeze}.txt")